# Задание 2. Восстановление данных кодом Хэмминга

In [1]:
import json
import random
import re
from pathlib import Path

import pandas as pd

## 1. Читаем результат Хаффмана

In [2]:
def project_path(relative_path):
    path = Path(relative_path)

    if path.exists():
        return path

    parent_path = Path("..") / path
    if parent_path.exists():
        return parent_path

    raise FileNotFoundError(f"Не найден файл: {relative_path}")


huffman_bin_path = project_path("data/huffman_block_1.bin")
huffman_meta_path = project_path("data/huffman_block_1.json")

huffman_meta = json.loads(huffman_meta_path.read_text(encoding="utf-8"))


def unpack_bits(packed_bytes, bit_length):
    bits = "".join(f"{byte:08b}" for byte in packed_bytes)
    return bits[:bit_length]


huffman_bits = unpack_bits(
    huffman_bin_path.read_bytes(),
    huffman_meta["bit_length"]
)

code_table = pd.DataFrame(huffman_meta["codes"], columns=["block", "code"])
decode_table = dict(zip(code_table["code"], code_table["block"]))

print("Файл Хаффмана:", huffman_bin_path)
print("Таблица Хаффмана:", huffman_meta_path)
print("Блок Хаффмана:", huffman_meta["block_size"], "символ")
print("Длина потока:", len(huffman_bits), "бит")
print("Размер файла:", huffman_bin_path.stat().st_size, "байт")
print("Размер таблицы:", huffman_meta_path.stat().st_size, "байт")

display(code_table.head())

Файл Хаффмана: ../data/huffman_block_1.bin
Таблица Хаффмана: ../data/huffman_block_1.json
Блок Хаффмана: 1 символ
Длина потока: 8794409 бит
Размер файла: 1099302 байт
Размер таблицы: 2069 байт


,block,code
0,e,000
1,P,0010000000
2,-,0010000001
3,!,001000001
4,G,0010000100


## 2. Проверяем, что таблица декодирования работает

In [3]:
text_path = project_path("data/brothers_karamazov.txt")


def remove_gutenberg_wrapper(text):
    start = re.search(r"\*\*\* START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text)
    end = re.search(r"\*\*\* END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text)

    if start:
        text = text[start.end():]
    if end:
        text = text[:end.start()]

    return text.strip()


source_text = remove_gutenberg_wrapper(
    text_path.read_text(encoding="utf-8").replace("\r\n", "\n").replace("\r", "\n")
)


def huffman_decode_from_table(bits, decode_table):
    buffer = ""
    restored_blocks = []

    for bit in bits:
        buffer += bit

        if buffer in decode_table:
            restored_blocks.append(decode_table[buffer])
            buffer = ""

    if buffer:
        raise ValueError("После декодирования остался незавершенный код")

    return "".join(restored_blocks)


huffman_decoded_text = huffman_decode_from_table(huffman_bits, decode_table)

print("Текст из Хаффмана восстановлен:", huffman_decoded_text == source_text)
assert huffman_decoded_text == source_text

Текст из Хаффмана восстановлен: True


## 3. Код Хэмминга `(7,4)`

In [4]:
def hamming74_encode_nibble(nibble):
    d1, d2, d3, d4 = (int(bit) for bit in nibble)

    p1 = d1 ^ d2 ^ d4
    p2 = d1 ^ d3 ^ d4
    p4 = d2 ^ d3 ^ d4

    return f"{p1}{p2}{d1}{p4}{d2}{d3}{d4}"


def hamming74_encode(bit_string):
    padding = (-len(bit_string)) % 4
    padded = bit_string + "0" * padding

    blocks = [
        hamming74_encode_nibble(padded[index:index + 4])
        for index in range(0, len(padded), 4)
    ]

    return "".join(blocks), padding


def hamming74_decode_block(block):
    bits = [int(bit) for bit in block]
    p1, p2, d1, p4, d2, d3, d4 = bits

    s1 = p1 ^ d1 ^ d2 ^ d4
    s2 = p2 ^ d1 ^ d3 ^ d4
    s4 = p4 ^ d2 ^ d3 ^ d4
    syndrome = s1 + 2 * s2 + 4 * s4

    if syndrome != 0:
        bits[syndrome - 1] ^= 1

    data = f"{bits[2]}{bits[4]}{bits[5]}{bits[6]}"
    return data, syndrome


def hamming74_decode(encoded_bits, padding):
    decoded_blocks = []
    corrected_blocks = 0

    for index in range(0, len(encoded_bits), 7):
        data, syndrome = hamming74_decode_block(encoded_bits[index:index + 7])
        decoded_blocks.append(data)
        corrected_blocks += int(syndrome != 0)

    decoded = "".join(decoded_blocks)
    if padding:
        decoded = decoded[:-padding]

    return decoded, corrected_blocks

## 4. Канал, попытка восстановления и сохранение файлов

In [5]:
OUTPUT_DIR = huffman_bin_path.parent
OUTPUT_DIR.mkdir(exist_ok=True)


def pack_bits(bit_string):
    padding = (-len(bit_string)) % 8
    padded = bit_string + "0" * padding

    packed = bytearray()
    for index in range(0, len(padded), 8):
        packed.append(int(padded[index:index + 8], 2))

    return bytes(packed), padding


def save_bit_stream(name, bit_string, extra_meta=None):
    bin_path = OUTPUT_DIR / f"{name}.bin"
    meta_path = OUTPUT_DIR / f"{name}.json"

    packed, padding = pack_bits(bit_string)
    bin_path.write_bytes(packed)

    meta = {
        "bit_length": len(bit_string),
        "padding_bits": padding,
    }
    if extra_meta:
        meta.update(extra_meta)

    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    return bin_path, meta_path


def read_bit_stream(bin_path, meta_path):
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    return unpack_bits(bin_path.read_bytes(), meta["bit_length"])


def flip_bit(bit):
    return "1" if bit == "0" else "0"


def noisy_channel_by_bits(bit_string, bit_error_probability=0.03, block_size=7, seed=42):
    rng = random.Random(seed)
    noisy = list(bit_string)
    errors_per_block = [0] * (len(bit_string) // block_size)
    flipped_bits = 0

    for position, bit in enumerate(noisy):
        if rng.random() < bit_error_probability:
            noisy[position] = flip_bit(bit)
            errors_per_block[position // block_size] += 1
            flipped_bits += 1

    stats = {
        "bit_error_probability": bit_error_probability,
        "flipped_bits": flipped_bits,
        "corrupted_blocks": sum(errors > 0 for errors in errors_per_block),
        "blocks_with_one_error": sum(errors == 1 for errors in errors_per_block),
        "blocks_with_multiple_errors": sum(errors >= 2 for errors in errors_per_block),
    }
    return "".join(noisy), stats


protected_bits, hamming_padding = hamming74_encode(huffman_bits)
noisy_bits, channel_stats = noisy_channel_by_bits(protected_bits, bit_error_probability=0.03)
restored_huffman_bits, corrected_blocks = hamming74_decode(noisy_bits, hamming_padding)

remaining_bit_errors = sum(
    original != restored
    for original, restored in zip(huffman_bits, restored_huffman_bits)
)
restored_successfully = restored_huffman_bits == huffman_bits

try:
    restored_text = huffman_decode_from_table(restored_huffman_bits, decode_table)
    text_restored = restored_text == source_text
    decode_error = None
except ValueError as error:
    restored_text = None
    text_restored = False
    decode_error = str(error)

protected_path, protected_meta_path = save_bit_stream(
    "hamming_protected",
    protected_bits,
    {"source": str(huffman_bin_path), "hamming_padding": hamming_padding}
)
noisy_path, noisy_meta_path = save_bit_stream(
    "hamming_noisy",
    noisy_bits,
    channel_stats
)
restored_path, restored_meta_path = save_bit_stream(
    "hamming_restored_huffman",
    restored_huffman_bits,
    {
        "source": "hamming_noisy.bin",
        "corrected_blocks": corrected_blocks,
        "remaining_bit_errors": remaining_bit_errors,
        "restored_successfully": restored_successfully,
        "text_restored": text_restored,
        "decode_error": decode_error,
    }
)

restored_bits_from_file = read_bit_stream(restored_path, restored_meta_path)
file_saved_correctly = restored_bits_from_file == restored_huffman_bits

print("Битов испортил канал:", channel_stats["flipped_bits"])
print("Блоков с одной ошибкой:", channel_stats["blocks_with_one_error"])
print("Блоков с 2+ ошибками:", channel_stats["blocks_with_multiple_errors"])
print("Блоков с ненулевым синдромом:", corrected_blocks)
print("Ошибок осталось в Хаффман-потоке:", remaining_bit_errors)
print("Файл hamming_restored_huffman сохранен без потерь:", file_saved_correctly)
print("Поток Хаффмана полностью восстановлен:", restored_successfully)
print("Текст полностью восстановлен:", text_restored)

assert file_saved_correctly


Битов испортил канал: 460819
Блоков с одной ошибкой: 384540
Блоков с 2+ ошибками: 37194
Блоков с ненулевым синдромом: 421361
Ошибок осталось в Хаффман-потоке: 64668
Файл hamming_restored_huffman сохранен без потерь: True
Поток Хаффмана полностью восстановлен: False
Текст полностью восстановлен: False


## Вывод

В этом варианте второе задание использует результат первого: сжатый Хаффманом файл и сохраненную таблицу кодов. Канал искажает каждый бит с вероятностью `3%`, поэтому в некоторых 7-битных блоках может появиться больше одной ошибки. Код Хэмминга `(7,4)` исправляет только одиночные ошибки, поэтому `hamming_restored_huffman.bin` является результатом попытки восстановления и не обязан совпадать с исходным Хаффман-потоком. Совпадение проверяется отдельной статистикой.